# E01

train a trigram language model, i.e. take two characters as an input to predict the 3rd one. Feel free to use either counting or a neural net. Evaluate the loss; Did it improve over a bigram model?

In [56]:
import torch
import torch.nn.functional as F

In [57]:
words = open("names.txt", 'r').read().splitlines()
words[0]

'emma'

In [58]:
chars = list('abcdefghijklmnopqrstuvwxyz')
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
print(list(stoi.items())[:5])
itos = {i:s for s,i in stoi.items()}
print(list(itos.items())[:5])

[('a', 1), ('b', 2), ('c', 3), ('d', 4), ('e', 5)]
[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd'), (5, 'e')]


In [59]:
x, y = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]
        x.append([ix1, ix2])
        y.append(ix3)
    #print(chs)
print(x)
print(y)

[[0, 5], [5, 13], [13, 13], [13, 1], [0, 15], [15, 12], [12, 9], [9, 22], [22, 9], [9, 1], [0, 1], [1, 22], [22, 1], [0, 9], [9, 19], [19, 1], [1, 2], [2, 5], [5, 12], [12, 12], [12, 1], [0, 19], [19, 15], [15, 16], [16, 8], [8, 9], [9, 1], [0, 3], [3, 8], [8, 1], [1, 18], [18, 12], [12, 15], [15, 20], [20, 20], [20, 5], [0, 13], [13, 9], [9, 1], [0, 1], [1, 13], [13, 5], [5, 12], [12, 9], [9, 1], [0, 8], [8, 1], [1, 18], [18, 16], [16, 5], [5, 18], [0, 5], [5, 22], [22, 5], [5, 12], [12, 25], [25, 14], [0, 1], [1, 2], [2, 9], [9, 7], [7, 1], [1, 9], [9, 12], [0, 5], [5, 13], [13, 9], [9, 12], [12, 25], [0, 5], [5, 12], [12, 9], [9, 26], [26, 1], [1, 2], [2, 5], [5, 20], [20, 8], [0, 13], [13, 9], [9, 12], [12, 1], [0, 5], [5, 12], [12, 12], [12, 1], [0, 1], [1, 22], [22, 5], [5, 18], [18, 25], [0, 19], [19, 15], [15, 6], [6, 9], [9, 1], [0, 3], [3, 1], [1, 13], [13, 9], [9, 12], [12, 1], [0, 1], [1, 18], [18, 9], [9, 1], [0, 19], [19, 3], [3, 1], [1, 18], [18, 12], [12, 5], [5, 20], [

In [60]:
num = len(x)
print(num)
x = torch.tensor(x)
y = torch.tensor(y)

196113


In [61]:
# initialize the 'network'
g = torch.Generator().manual_seed(2147483647)
W = torch.rand((54, 27), generator=g, requires_grad=True)

In [62]:
xenc = F.one_hot(x, num_classes=(27)).float()
xenc = xenc.view(xenc.shape[0], -1)
#print(xenc[0])

# gradient descent
for k in range(100):
    
    #forward pass
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)
    loss = -probs[torch.arange(num), y].log().mean() + 0.01*(W**2).mean()
    print(loss.item())
    
    #backward pass
    W.grad = None
    loss.backward()
    
    # update
    W.data += -50 * W.grad

3.4310197830200195
2.8565454483032227
2.670452356338501
2.575190544128418
2.517145872116089
2.478353977203369
2.4504599571228027
2.429287910461426
2.412524938583374
2.3988399505615234
2.387404441833496
2.3776803016662598
2.3692944049835205
2.3619801998138428
2.355538845062256
2.3498194217681885
2.344703197479248
2.340097188949585
2.3359262943267822
2.33212947845459
2.328657388687134
2.325468063354492
2.322528123855591
2.3198070526123047
2.317281484603882
2.3149306774139404
2.3127360343933105
2.310682773590088
2.3087563514709473
2.306946277618408
2.305241823196411
2.303633213043213
2.3021128177642822
2.300673723220825
2.299309253692627
2.298013687133789
2.2967822551727295
2.295609712600708
2.29449200630188
2.2934257984161377
2.2924065589904785
2.2914323806762695
2.2904999256134033
2.2896065711975098
2.2887494564056396
2.2879271507263184
2.2871367931365967
2.286377429962158
2.285646438598633
2.284942626953125
2.284264326095581
2.2836105823516846
2.282979726791382
2.2823708057403564
2.281

The loss of the bigram model was around 2.49, while the trigram model achieved a lower loss of around 2.27. Therefore, the trigram model performs slightly better. This suggests that using two previous characters provides useful additional context for predicting the next character.

In [63]:
g = torch.Generator().manual_seed(2147483647)

for i in range(5):

    out = []

    ix1 = 0  # first character: '.'
    ix2 = 0  # second character: '.'

    while True:

        xenc = F.one_hot(torch.tensor([[ix1, ix2]]), num_classes=27).float()
        xenc = xenc.view(1, -1)

        # forward pass
        logits = xenc @ W
        counts = logits.exp()
        p = counts / counts.sum(1, keepdims=True)

        # sample next character
        ix3 = torch.multinomial(
            p,
            num_samples=1,
            replacement=True,
            generator=g
        ).item()

        if ix3 == 0:
            break

        out.append(itos[ix3])

        # shift the context
        ix1 = ix2
        ix2 = ix3

    print(''.join(out))

dexze
iogh
urailazitynn
vinish
na


# E02

split up the dataset randomly into 80% train set, 10% dev set, 10% test set. Train the bigram and trigram models only on the training set. Evaluate them on dev and test splits. What can you see?

In [87]:
import random

random.seed(42)
random.shuffle(words)

n = len(words)

training_set = words[:int(0.8*n)]
dev_set  = words[int(0.8 * n):int(0.9 * n)]
test_set  = words[int(0.9 * n):]

In [88]:
def build_dataset(words, n):
    x, y = [], []

    for w in words:
        chs = ['.'] + list(w) + ['.']

        if n == 2:  # bigram
            for ch1, ch2 in zip(chs, chs[1:]):
                ix1 = stoi[ch1]
                ix2 = stoi[ch2]

                x.append(ix1)
                y.append(ix2)

        elif n == 3:  # trigram
            for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
                ix1 = stoi[ch1]
                ix2 = stoi[ch2]
                ix3 = stoi[ch3]

                x.append([ix1, ix2])
                y.append(ix3)

    return torch.tensor(x), torch.tensor(y)

In [89]:
Xtr, Ytr = build_dataset(training_set, 2)
Xdev, Ydev = build_dataset(dev_set, 2)
Xte, Yte = build_dataset(test_set, 2)

## Bigram Model

In [90]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)
xtr_enc = F.one_hot(Xtr, num_classes=27).float()

for k in range(100):
  
  logits = xtr_enc @ W
  counts = logits.exp()
  probs = counts / counts.sum(1, keepdims=True)
  loss = -probs[torch.arange(Ytr.shape[0]), Ytr].log().mean() + 0.01*(W**2).mean()
  print(loss.item())
  
  W.grad = None
  loss.backward()
  
  W.data += -50 * W.grad

3.7687532901763916
3.3775827884674072
3.1594560146331787
3.025578737258911
2.9329609870910645
2.865790367126465
2.8152647018432617
2.7757887840270996
2.743925094604492
2.7175352573394775
2.69524884223938
2.6761574745178223
2.6596338748931885
2.645223379135132
2.6325786113739014
2.6214237213134766
2.611534357070923
2.6027235984802246
2.594836473464966
2.587742805480957
2.581333637237549
2.5755186080932617
2.5702199935913086
2.5653746128082275
2.560926914215088
2.556830883026123
2.5530476570129395
2.5495431423187256
2.5462896823883057
2.543262243270874
2.5404396057128906
2.5378034114837646
2.535337448120117
2.533027172088623
2.53085994720459
2.5288245677948
2.5269105434417725
2.525108814239502
2.5234103202819824
2.5218076705932617
2.520294427871704
2.5188634395599365
2.5175092220306396
2.516225814819336
2.5150084495544434
2.5138535499572754
2.5127553939819336
2.5117111206054688
2.5107169151306152
2.5097696781158447
2.508866310119629
2.5080039501190186
2.5071797370910645
2.506392240524292

In [91]:
# Dev Evaluation
xdev_enc = F.one_hot(Xdev, num_classes=27).float()
logits = xdev_enc @ W
counts = logits.exp()
probs = counts / counts.sum(1, keepdims=True)
dev_loss = -probs[torch.arange(Ydev.shape[0]), Ydev].log().mean()

print("dev loss:", dev_loss.item())

dev loss: 2.4777820110321045


In [92]:
# Test Evaluation
xtest_enc = F.one_hot(Xte, num_classes=27).float()
logits = xtest_enc @ W
counts = logits.exp()
probs = counts / counts.sum(1, keepdims=True)
test_loss = -probs[torch.arange(Yte.shape[0]), Yte].log().mean()

print("test loss:", test_loss.item())

test loss: 2.4817264080047607


## Trigram Model

In [93]:
Xtr, Ytr = build_dataset(training_set, 3)
Xdev, Ydev = build_dataset(dev_set, 3)
Xte, Yte = build_dataset(test_set, 3)

In [94]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((54, 27), generator=g, requires_grad=True)

xtr_enc = F.one_hot(Xtr, num_classes=27).float()
xtr_enc = xtr_enc.view(xtr_enc.shape[0], -1)

for k in range(100):

    logits = xtr_enc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdims=True)
    loss = -probs[torch.arange(Ytr.shape[0]),Ytr].log().mean() + 0.01 * (W**2).mean()

    print(loss.item())

    W.grad = None
    loss.backward()

    W.data += -50 * W.grad

4.197098255157471
3.3641514778137207
3.0486741065979004
2.877676486968994
2.7731878757476807
2.7005131244659424
2.6447935104370117
2.600588321685791
2.564535617828369
2.534698486328125
2.5096631050109863
2.4884634017944336
2.470320463180542
2.454648971557617
2.440969705581665
2.4289190769195557
2.418208599090576
2.408618211746216
2.3999741077423096
2.3921403884887695
2.3850064277648926
2.378484010696411
2.37249755859375
2.3669850826263428
2.361894130706787
2.3571784496307373
2.3528008460998535
2.3487257957458496
2.3449254035949707
2.3413736820220947
2.338048219680786
2.334928512573242
2.331998109817505
2.329240560531616
2.3266420364379883
2.324190139770508
2.3218729496002197
2.319680690765381
2.3176043033599854
2.3156349658966064
2.3137643337249756
2.3119871616363525
2.310295581817627
2.3086843490600586
2.307147979736328
2.3056812286376953
2.3042805194854736
2.3029403686523438
2.3016581535339355
2.3004298210144043
2.2992517948150635
2.298121452331543
2.2970359325408936
2.29599261283874

In [ ]:
# Dev Evaluation
xdev_enc = F.one_hot(Xdev, num_classes=27).float()
xdev_enc = xdev_enc.view(xdev_enc.shape[0], -1)

logits = xdev_enc @ W

counts = logits.exp()

probs = counts / counts.sum(1, keepdims=True)

dev_loss = -probs[torch.arange(Ydev.shape[0]),Ydev].log().mean()

print("Trigram dev loss:", dev_loss.item())

Trigram dev loss: 2.273601531982422


In [96]:
# Test Evaluation
xtest_enc = F.one_hot(Xte, num_classes=27).float()
xtest_enc = xtest_enc.view(xtest_enc.shape[0], -1)

logits = xtest_enc @ W

counts = logits.exp()

probs = counts / counts.sum(1, keepdims=True)

test_loss = -probs[torch.arange(Yte.shape[0]),Yte].log().mean()

print("Trigram test loss:", test_loss.item())

Trigram test loss: 2.270625591278076


The trigram model performs significantly better than the bigram model. Its loss is around 2.27 on the train, dev, and test sets, compared to around 2.48 for the bigram model. This shows that using two previous characters provides useful additional context for predicting the next character.

The train, dev, and test losses are also very close for both models, so there is no significant evidence of overfitting.

# E03

use the dev set to tune the strength of smoothing (or regularization) for the trigram model - i.e. try many possibilities and see which one works best based on the dev set loss. What patterns can you see in the train and dev set loss as you tune this strength? Take the best setting of the smoothing and evaluate on the test set once and at the end. How good of a loss do you achieve?

In [100]:
regs = [0, 0.0001, 0.001, 0.01, 0.1, 1]
for reg in regs:
    g = torch.Generator().manual_seed(2147483647)
    W = torch.randn((54, 27), generator=g, requires_grad = True)
    
    xtr_enc = F.one_hot(Xtr, num_classes=27).float()
    xtr_enc = xtr_enc.view(xtr_enc.shape[0], -1)
    
    for k in range(100):

        # forward
        logits = xtr_enc @ W
        counts = logits.exp()
        probs = counts / counts.sum(1, keepdims=True)

        loss = -probs[torch.arange(Ytr.shape[0]), Ytr].log().mean() + reg * (W**2).mean()

        # backward
        W.grad = None
        loss.backward()

        # update
        W.data += -50 * W.grad

    # train loss WITHOUT regularization
    train_loss = -probs[torch.arange(Ytr.shape[0]), Ytr].log().mean()

    # dev
    xdev_enc = F.one_hot(Xdev, num_classes=27).float()
    xdev_enc = xdev_enc.view(xdev_enc.shape[0], -1)

    logits = xdev_enc @ W
    counts = logits.exp()
    probs_dev = counts / counts.sum(1, keepdims=True)

    dev_loss = -probs_dev[torch.arange(Ydev.shape[0]), Ydev].log().mean()

    print(
        f"reg={reg:<8} "
        f"train={train_loss.item():.4f} "
        f"dev={dev_loss.item():.4f}"
    )

reg=0        train=2.2622 dev=2.2734
reg=0.0001   train=2.2622 dev=2.2734
reg=0.001    train=2.2622 dev=2.2734
reg=0.01     train=2.2625 dev=2.2736
reg=0.1      train=2.2722 dev=2.2820
reg=1        train=2.3925 dev=2.3987


As the regularization strength increases, the training loss gradually increases because the model is increasingly constrained to keep its weights small. For very small regularization strengths (0 to 0.001), there is almost no difference in either the training or dev loss. Stronger regularization, such as 0.1 and 1, makes both losses worse.

The train and dev losses remain very close across all settings, suggesting that the model is not suffering from significant overfitting. In this experiment, regularization does not improve generalization; the best dev loss is achieved with essentially no regularization (`reg=0`).

In [101]:
# train final model with reg = 0
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((54, 27), generator=g, requires_grad=True)

xtr_enc = F.one_hot(Xtr, num_classes=27).float()
xtr_enc = xtr_enc.view(xtr_enc.shape[0], -1)

reg = 0

for k in range(100):

    logits = xtr_enc @ W
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdims=True)

    loss = -probs[
        torch.arange(Ytr.shape[0]), Ytr].log().mean() + reg * (W**2).mean()

    W.grad = None
    loss.backward()

    W.data += -50 * W.grad

# Test Evaluation
xtest_enc = F.one_hot(Xte, num_classes=27).float()
xtest_enc = xtest_enc.view(xtest_enc.shape[0], -1)

logits = xtest_enc @ W
counts = logits.exp()
probs = counts / counts.sum(1, keepdims=True)

test_loss = -probs[torch.arange(Yte.shape[0]), Yte].log().mean()

print("Final test loss:", test_loss.item())

Final test loss: 2.2704546451568604


# E04

we saw that our 1-hot vectors merely select a row of W, so producing these vectors explicitly feels wasteful. Can you delete our use of F.one_hot in favor of simply indexing into rows of W?


In [108]:
Xtr_bi, Ytr_bi = build_dataset(training_set, 2)
Xdev_bi, Ydev_bi = build_dataset(dev_set, 2)
Xte_bi, Yte_bi = build_dataset(test_set, 2)

g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

for k in range(100):
  logits = W[Xtr_bi]
  counts = logits.exp()
  probs = counts / counts.sum(1, keepdims=True)
  loss = -probs[torch.arange(Ytr_bi.shape[0]), Ytr_bi].log().mean() + 0.01*(W**2).mean()
  print(loss.item())
  
  W.grad = None
  loss.backward()
  
  W.data += -50 * W.grad

3.7687532901763916
3.3775784969329834
3.1594512462615967
3.0255773067474365
2.9329607486724854
2.8657898902893066
2.8152644634246826
2.7757887840270996
2.743925094604492
2.7175354957580566
2.69524884223938
2.6761574745178223
2.6596343517303467
2.645223379135132
2.6325786113739014
2.6214237213134766
2.611534595489502
2.6027235984802246
2.5948362350463867
2.587742805480957
2.581333637237549
2.5755181312561035
2.5702202320098877
2.5653743743896484
2.560926914215088
2.556830883026123
2.5530476570129395
2.5495431423187256
2.5462896823883057
2.543262243270874
2.5404393672943115
2.5378031730651855
2.535337448120117
2.533027172088623
2.53085994720459
2.528824806213379
2.5269105434417725
2.525108814239502
2.5234103202819824
2.52180814743042
2.520294666290283
2.5188634395599365
2.5175092220306396
2.516225814819336
2.5150084495544434
2.513853073120117
2.5127553939819336
2.5117111206054688
2.5107169151306152
2.5097696781158447
2.508866310119629
2.5080039501190186
2.5071797370910645
2.5063922405242

# E05

look up and use F.cross_entropy instead. You should achieve the same result. Can you think of why we'd prefer to use F.cross_entropy instead?

In [109]:
Xtr_bi, Ytr_bi = build_dataset(training_set, 2)
Xdev_bi, Ydev_bi = build_dataset(dev_set, 2)
Xte_bi, Yte_bi = build_dataset(test_set, 2)

g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

for k in range(100):
  logits = W[Xtr_bi]
  loss = F.cross_entropy(logits, Ytr_bi) + 0.01 * (W**2).mean()
  print(loss.item())
  
  W.grad = None
  loss.backward()
  
  W.data += -50 * W.grad

3.7687532901763916
3.3775787353515625
3.1594512462615967
3.0255777835845947
2.9329609870910645
2.8657898902893066
2.8152644634246826
2.7757887840270996
2.743925094604492
2.7175354957580566
2.695249080657959
2.6761574745178223
2.6596341133117676
2.645223379135132
2.6325788497924805
2.6214234828948975
2.611534357070923
2.6027235984802246
2.594836473464966
2.587742567062378
2.5813333988189697
2.5755186080932617
2.5702202320098877
2.5653746128082275
2.560927152633667
2.556831121444702
2.5530474185943604
2.5495433807373047
2.5462899208068848
2.543262004852295
2.5404393672943115
2.5378031730651855
2.535337448120117
2.533027172088623
2.530860185623169
2.5288245677948
2.5269107818603516
2.525109052658081
2.5234105587005615
2.521807909011841
2.520294427871704
2.5188634395599365
2.5175087451934814
2.516226053237915
2.5150084495544434
2.513853073120117
2.5127556324005127
2.511711359024048
2.510716676712036
2.5097696781158447
2.508866310119629
2.5080039501190186
2.5071799755096436
2.50639200210571

As we can see, the loss values are almost the same. We prefer F.cross_entropy because it is simpler, numerically more stable, and more computationally efficient than manually computing exp, normalizing into probabilities, and taking the log.